# Master Data Alat Berat: Standardisasi, Entity Resolution & Golden Record
### *Heavy Equipment Master Data Management — Data Quality, Golden Record & Price-Model Impact (CRISP-DM)*

**Ringkasan.** Data transaksi lelang 412 ribu unit alat berat (Kaggle *Blue Book for Bulldozers*) diperlakukan
seperti data aset armada perusahaan kontraktor tambang: diprofilkan, diuji dengan **aturan kualitas data**,
distandardisasi, dideduplikasi, lalu dibentuk menjadi **satu golden record per mesin** dan **satu master per
model**, disimpan ke **database SQL** (skema master–transaksi), dan dibuktikan dampaknya pada **model harga**.

| Tahap CRISP-DM | Isi |
|---|---|
| 1. Business Understanding | mengapa master data alat berat penting, tujuan, kriteria sukses |
| 2. Data Understanding | profiling 4 dimensi kualitas (kelengkapan, keunikan, validitas, konsistensi), matriks *applicability*, anomali |
| 3. Data Preparation | katalog aturan DQ, standardisasi, perbaikan korupsi Excel, master model, *entity resolution*, golden record, scorecard sebelum–sesudah, integrasi batch baru, SQLite |
| 4. Modeling | model harga (Ridge & Gradient Boosting) dengan data mentah vs data master, tanpa kebocoran waktu |
| 5. Evaluation | bootstrap CI, error per segmen, ablation, distorsi KPI bisnis, kriteria sukses, rekomendasi tata kelola data |

**Cara menjalankan:** letakkan `Train.csv`, `Valid.csv`, `Test.csv` di folder `data/` (atau folder yang sama dengan
notebook), lalu *Run All*. Waktu jalan ± 8–12 menit. Library: `pandas numpy scikit-learn matplotlib` (+ `sqlite3` bawaan Python).

**Output:** `outputs/heavy_equipment_master.db` (SQLite) dan `outputs/*.csv` (scorecard & hasil aturan — siap untuk dashboard Power BI).

In [ ]:
import re
import sqlite3
import time
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 200)
pd.set_option("display.float_format", lambda v: f"{v:,.4f}")
T0 = time.time()

# ---- gaya visual konsisten: palet ramah buta warna, grid tipis, teks netral
C = {"blue": "#2a78d6", "orange": "#eb6834", "green": "#1baf7a", "yellow": "#eda100", "purple": "#6250d6",
     "ink": "#0b0b0b", "ink2": "#52514e", "muted": "#898781", "grid": "#e1e0d9",
     "good": "#0ca30c", "warn": "#fab219", "bad": "#d03b3b"}
plt.rcParams.update({
    "figure.dpi": 110, "figure.facecolor": "white", "axes.facecolor": "white",
    "axes.spines.top": False, "axes.spines.right": False, "axes.edgecolor": "#c3c2b7",
    "axes.grid": True, "grid.color": C["grid"], "grid.linewidth": 0.6, "axes.axisbelow": True,
    "axes.titlesize": 12, "axes.titleweight": "bold", "axes.titlelocation": "left", "axes.titlepad": 20,
    "axes.labelcolor": C["ink2"], "xtick.color": C["muted"], "ytick.color": C["muted"],
    "xtick.labelcolor": C["ink2"], "ytick.labelcolor": C["ink2"], "legend.frameon": False,
    "lines.linewidth": 1.8, "font.size": 10,
})


def subtitle(ax, text):
    """Kalimat kesimpulan kecil di bawah judul grafik."""
    ax.text(0, 1.015, text, transform=ax.transAxes, fontsize=9, color=C["ink2"], va="bottom")


def pct(v):
    return f"{v:.1%}"


CONFIG = {
    "DATA_DIR": Path("data") if Path("data/Train.csv").exists() else Path("."),
    "OUT_DIR": Path("outputs"),
    "YEAR_MIN": 1950,                 # tahun pembuatan minimum yang masuk akal
    "HOURS_MAX": 100_000,             # jam operasi maksimum seumur hidup alat berat
    "HOURS_PER_YEAR_MAX": 8_760,      # 24 jam × 365 hari
    "APPLICABLE_SHARE": 0.50,         # atribut dianggap berlaku utk grup produk bila terisi ≥ 50%
    "ML_CUTOFF": pd.Timestamp("2011-01-01"),   # latih < 2011, uji = penjualan tahun 2011
    "SEED": 42,
}
CONFIG["OUT_DIR"].mkdir(exist_ok=True)
SENTINELS = {"None or Unspecified", "#NAME?", "", "nan"}

## 1. Business Understanding

**Konteks.** Bagi kontraktor tambang, alat berat (excavator, dozer, wheel loader, motor grader) adalah aset
terbesar. Hampir semua keputusan — rencana perawatan, alokasi unit ke site, depresiasi, *buy/sell/rebuild*,
pengadaan suku cadang — bergantung pada **master data unit**: model apa, tahun berapa, spesifikasi apa,
sudah berapa jam operasi. Master data yang kotor menyebabkan:
* **Unit ganda atau tertukar** — satu nomor unit dipakai untuk dua alat berbeda → riwayat perawatan tercampur.
* **Tahun pembuatan salah** → umur aset, depresiasi, dan valuasi salah.
* **Deskripsi model tidak seragam** (`D155A-1` vs `D155A1`) → laporan per model terpecah, stok suku cadang salah.
* **Hour meter tidak masuk akal** (turun, nol, jutaan jam) → jadwal servis berbasis jam tidak dapat diandalkan.

**Dataset.** *Blue Book for Bulldozers* (Kaggle): riwayat lelang alat berat 1989–2012 — dipakai sebagai proksi data
armada multi-sumber. `Train.csv` (dengan harga) adalah data historis; `Valid.csv` & `Test.csv` (2012, tanpa harga)
diperlakukan sebagai **batch data baru** yang harus diintegrasikan ke master.

**Tujuan**
1. Mengukur kualitas data secara objektif (profiling 4 dimensi + aturan kualitas data).
2. Menstandardisasi & mendeduplikasi master model, dan membentuk **satu golden record per mesin** yang dapat ditelusuri ke sumbernya.
3. Menyimpan hasil ke database SQL dengan skema master–transaksi.
4. Membuktikan dampak bisnis lewat model harga (*valuation model*).

**Kriteria sukses (ditetapkan sebelum analisis)**
| # | Kriteria | Target |
|---|---|---|
| S1 | Skor validitas, konsistensi, dan keunikan setelah pembersihan | ≥ 99% per dimensi |
| S2 | Kelengkapan naik tanpa imputasi palsu (pembersihan tidak boleh "mengarang" data) | naik, dilaporkan apa adanya |
| S3 | Golden record: satu nilai per atribut per mesin & per model | 100% |
| S4 | Ketertelusuran: setiap transaksi & ID sumber terhubung ke golden record (lolos cek foreign key) | 100% |
| S5 | Model harga dengan data master lebih akurat (RMSLE) dan signifikan (CI 95%) | minimal pada model valuasi linear |
| S6 | Model *gradient boosting* tidak menjadi lebih buruk | selisih ≤ noise antar-seed |

## 2. Data Understanding
### 2.1 Memuat data & inventaris kolom

In [ ]:
def load(name):
    df = pd.read_csv(CONFIG["DATA_DIR"] / name, low_memory=False)
    df["saledate"] = pd.to_datetime(df["saledate"], format="%m/%d/%Y %H:%M")
    df["source_batch"] = name.replace(".csv", "")
    return df


train_raw, valid_raw, test_raw = load("Train.csv"), load("Valid.csv"), load("Test.csv")
for n, d in (("Train", train_raw), ("Valid", valid_raw), ("Test", test_raw)):
    print(f"{n:5s}: {len(d):>7,} baris | {d['saledate'].min():%Y-%m-%d} → {d['saledate'].max():%Y-%m-%d} "
          f"| ada SalePrice: {'SalePrice' in d}")

SPEC_COLS = list(train_raw.loc[:, "Drive_System":"Steering_Controls"].columns)
COLUMN_GROUPS = {
    "Identitas": ["SalesID", "MachineID", "ModelID"],
    "Transaksi": ["SalePrice", "saledate", "auctioneerID", "datasource", "state"],
    "Atribut mesin": ["YearMade", "MachineHoursCurrentMeter", "UsageBand"],
    "Atribut model": ["fiModelDesc", "fiBaseModel", "fiSecondaryDesc", "fiModelSeries", "fiModelDescriptor",
                      "ProductSize", "fiProductClassDesc", "ProductGroup", "ProductGroupDesc"],
    "Spesifikasi/konfigurasi": SPEC_COLS,
}
pd.DataFrame([(g, len(c), ", ".join(c[:4]) + (" …" if len(c) > 4 else "")) for g, c in COLUMN_GROUPS.items()],
             columns=["Kelompok", "Jumlah kolom", "Contoh"])

### 2.2 Matriks *applicability*: nilai kosong yang "wajar" vs yang benar-benar hilang
Kolom spesifikasi tampak 70–94% kosong. Namun pola kosongnya **struktural**: `Blade_Width` hanya relevan untuk
motor grader, `Stick_Length` hanya untuk excavator. Mengukur kelengkapan tanpa memperhitungkan hal ini akan
menghasilkan skor yang menyesatkan. Atribut dianggap **berlaku** bagi suatu grup produk bila terisi ≥ 50% di grup tersebut.

In [ ]:
populated = train_raw.groupby("ProductGroup")[SPEC_COLS].apply(lambda d: d.notna().mean()).T
APPLICABLE = populated >= CONFIG["APPLICABLE_SHARE"]          # attr × ProductGroup → bool
GROUP_LABEL = train_raw.groupby("ProductGroup")["ProductGroupDesc"].first().to_dict()

fig, ax = plt.subplots(figsize=(9, 10))
from matplotlib.colors import LinearSegmentedColormap
seq = LinearSegmentedColormap.from_list("seq", ["#f4f8fd", "#86b6ef", "#256abf", "#0d366b"])
im = ax.imshow(populated.values, cmap=seq, vmin=0, vmax=1, aspect="auto")
ax.set_xticks(range(populated.shape[1]), [f"{g}\n{GROUP_LABEL[g]}" for g in populated.columns], fontsize=8)
ax.set_yticks(range(populated.shape[0]), populated.index, fontsize=8.5)
for i in range(populated.shape[0]):
    for j in range(populated.shape[1]):
        v = populated.iat[i, j]
        if v > 0.02:
            ax.text(j, i, f"{v:.0%}", ha="center", va="center", fontsize=7,
                    color="white" if v > 0.6 else C["ink"])
ax.grid(visible=False)
fig.colorbar(im, ax=ax, fraction=0.03, label="Proporsi terisi")
ax.set_title("Matriks applicability atribut spesifikasi × grup produk")
subtitle(ax, "Setiap atribut hanya berlaku untuk 1–2 grup produk → kosong di grup lain = 'N/A', bukan data hilang")
plt.tight_layout()
plt.show()

### 2.3 Profiling 4 dimensi kualitas data per kolom
* **Kelengkapan (completeness)** — terisi dan bukan nilai *sentinel* (`None or Unspecified`, `#NAME?`, `YearMade = 1000`, jam = 0),
  dihitung **hanya pada baris di mana atribut berlaku**.
* **Keunikan (uniqueness)** — untuk kolom identitas: proporsi nilai yang tidak duplikat; untuk lainnya: jumlah nilai distinct (informasi).
* **Validitas (validity)** — nilai mengikuti domain/format yang diharapkan.
* **Konsistensi (consistency)** — atribut yang seharusnya tetap untuk satu entitas (mesin/model) hanya punya satu nilai.

In [ ]:
EXCEL_SCI = r"^-?\d+(\.\d+)?E[+-]\d+$"
VALIDATORS = {      # fungsi -> Series bool (True = valid) untuk nilai yang terisi
    "YearMade": lambda d: d["YearMade"].between(CONFIG["YEAR_MIN"], d["saledate"].dt.year),
    "MachineHoursCurrentMeter": lambda d: d["MachineHoursCurrentMeter"].between(1, CONFIG["HOURS_MAX"]),
    "SalePrice": lambda d: d["SalePrice"] > 0,
    "saledate": lambda d: d["saledate"].between("1989-01-01", "2012-12-31"),
    "Enclosure": lambda d: d["Enclosure"].isin(["OROPS", "EROPS", "EROPS w AC", "NO ROPS"]),
    "Tire_Size": lambda d: d["Tire_Size"].astype("string").str.fullmatch(r"\d+(\.\d+)?"),
    "Undercarriage_Pad_Width": lambda d: d["Undercarriage_Pad_Width"].astype("string").str.fullmatch(r"\d+(\.\d+)? inch"),
    "Stick_Length": lambda d: d["Stick_Length"].astype("string").str.fullmatch(r"\d+' \d+\""),
    "Blade_Width": lambda d: d["Blade_Width"].astype("string").str.fullmatch(r"\d+'"),
    "fiProductClassDesc": lambda d: d["fiProductClassDesc"].astype("string").str.fullmatch(r".+ - [\d.]+ to [\d.]+ .+"),
}
MACHINE_LEVEL = ["ModelID", "YearMade", "fiModelDesc", "ProductGroup"]
MODEL_LEVEL = ["fiModelDesc", "fiBaseModel", "fiProductClassDesc", "ProductGroup", "ProductSize"]


def is_missing(s: pd.Series, col: str) -> pd.Series:
    m = s.isna()
    if s.dtype == object or str(s.dtype) in ("str", "string"):
        m |= s.astype("string").str.strip().isin(list(SENTINELS)).fillna(False)
    if col == "YearMade":
        m |= s.eq(1000)
    if col == "MachineHoursCurrentMeter":
        m |= s.eq(0)
    return m


def applicable_mask(df: pd.DataFrame, col: str) -> pd.Series:
    if col in APPLICABLE.index:
        return df["ProductGroup"].map(APPLICABLE.loc[col]).fillna(False).astype(bool)
    return pd.Series(True, index=df.index)


def entity_consistency(df, entity, col):
    d = df.loc[~is_missing(df[col], col), [entity, col]]
    n = d.groupby(entity)[col].nunique()
    return (n <= 1).mean() if len(n) else np.nan


def profile(df: pd.DataFrame, machine_key="MachineID", model_key="ModelID", cols=None) -> pd.DataFrame:
    rows = []
    for col in cols or [c for c in train_raw.columns if c != "source_batch"]:
        s = df[col]
        app = applicable_mask(df, col)
        miss = is_missing(s, col)
        filled = s[app & ~miss]
        if col in VALIDATORS and len(filled):
            validity = VALIDATORS[col](df.loc[filled.index]).fillna(False).mean()
        elif s.dtype == object or str(s.dtype) in ("str", "string"):
            validity = 1 - filled.astype("string").str.match(EXCEL_SCI).fillna(False).mean() if len(filled) else np.nan
        else:
            validity = np.nan
        cons = []
        if col in MACHINE_LEVEL:
            cons.append(entity_consistency(df, machine_key, col))
        if col in MODEL_LEVEL and col != "ProductGroup":
            cons.append(entity_consistency(df, model_key, col))
        rows.append({
            "kolom": col,
            "kelengkapan": (app & ~miss).sum() / max(app.sum(), 1),
            "kelengkapan_naif": (~s.isna()).mean(),
            "distinct": s.nunique(),
            "keunikan_id": (1 - s.duplicated(keep=False).mean()) if col in ("SalesID", "MachineID") else np.nan,
            "validitas": validity,
            "konsistensi": np.nanmin(cons) if cons else np.nan,
        })
    return pd.DataFrame(rows).set_index("kolom")


prof_raw = profile(train_raw)
prof_raw.sort_values("kelengkapan").head(20)

In [ ]:
show = ["YearMade", "MachineHoursCurrentMeter", "UsageBand", "ModelID", "fiModelDesc", "fiProductClassDesc",
        "ProductSize", "Enclosure", "Hydraulics", "Tire_Size", "Undercarriage_Pad_Width", "Stick_Length",
        "Blade_Width", "Ride_Control", "Coupler"]
mat = prof_raw.loc[show, ["kelengkapan", "validitas", "konsistensi"]]
fig, axes = plt.subplots(1, 2, figsize=(14, 6.5), gridspec_kw={"width_ratios": [1.1, 1]})
ax = axes[0]
rg = LinearSegmentedColormap.from_list("rg", ["#d03b3b", "#fab219", "#f4f4f2", "#86c98a", "#0ca30c"])
im = ax.imshow(mat.values, cmap=rg, vmin=0.5, vmax=1, aspect="auto")
ax.set_xticks(range(3), ["Kelengkapan\n(atribut berlaku)", "Validitas", "Konsistensi\n(per entitas)"], fontsize=9)
ax.set_yticks(range(len(mat)), mat.index, fontsize=9)
for i in range(mat.shape[0]):
    for j in range(mat.shape[1]):
        v = mat.iat[i, j]
        ax.text(j, i, "—" if pd.isna(v) else f"{v:.0%}", ha="center", va="center", fontsize=8.5, color=C["ink"])
ax.grid(visible=False)
ax.set_title("Profil kualitas kolom kunci (data mentah)")
subtitle(ax, "Merah = masalah; '—' = dimensi tidak relevan untuk kolom tsb.")
ax = axes[1]
sp = prof_raw.loc[SPEC_COLS, ["kelengkapan_naif", "kelengkapan"]].sort_values("kelengkapan")
y_ = np.arange(len(sp))
ax.barh(y_ + 0.2, sp["kelengkapan_naif"], height=0.4, color=C["muted"], label="Naif (semua baris)")
ax.barh(y_ - 0.2, sp["kelengkapan"], height=0.4, color=C["blue"], label="Hanya baris yang berlaku")
ax.set_yticks(y_, sp.index, fontsize=8)
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.0%}"))
ax.legend(loc="lower right", fontsize=8)
ax.set_title("Kelengkapan atribut spesifikasi: naif vs berbasis applicability")
subtitle(ax, "Kesenjangan nyata = 'None or Unspecified' pada atribut yang BERLAKU")
ax.grid(axis="y", visible=False)
plt.tight_layout()
plt.show()

### 2.4 Anomali utama
#### (a) Tahun pembuatan & jam operasi

In [ ]:
ym = train_raw["YearMade"]
hrs = train_raw["MachineHoursCurrentMeter"]
age_known = train_raw["saledate"].dt.year - ym.where(ym.between(CONFIG["YEAR_MIN"], 2012))
anom = pd.DataFrame([
    ("YearMade = 1000 (placeholder)", (ym == 1000).sum()),
    ("YearMade 1001–1949 (tidak masuk akal)", ym.between(1001, 1949).sum()),
    ("YearMade > tahun penjualan", (ym > train_raw["saledate"].dt.year).sum()),
    ("Jam operasi kosong", hrs.isna().sum()),
    ("Jam operasi = 0 (placeholder)", (hrs == 0).sum()),
    (f"Jam operasi > {CONFIG['HOURS_MAX']:,}", (hrs > CONFIG["HOURS_MAX"]).sum()),
    ("Jam per tahun umur > 8.760 (mustahil)", ((hrs / age_known.clip(lower=1)) > CONFIG["HOURS_PER_YEAR_MAX"]).sum()),
], columns=["Anomali", "Jumlah baris"])
anom["% baris"] = anom["Jumlah baris"] / len(train_raw)
display(anom)

fig, axes = plt.subplots(1, 2, figsize=(14, 4.2))
ax = axes[0]
vc = ym.value_counts().sort_index()
ax.bar(vc.index.astype(str).where(vc.index > 1000, "1000"), vc.values, color=[C["bad"] if v < 1950 else C["blue"] for v in vc.index])
ax.set_xticks(range(0, len(vc), 8))
ax.set_xticklabels(vc.index.astype(str)[::8], rotation=45, fontsize=8)
ax.set_ylabel("Jumlah transaksi")
ax.set_title("Distribusi YearMade")
subtitle(ax, f"{(ym == 1000).mean():.1%} transaksi memakai placeholder 1000 (merah = tidak valid)")
ax.grid(axis="x", visible=False)
ax = axes[1]
h = hrs.dropna()
bins = np.logspace(0, 7, 60)
ax.hist(h[h > 0], bins=bins, color=C["blue"], alpha=0.85)
ax.set_xscale("log")
ax.axvline(CONFIG["HOURS_MAX"], color=C["bad"], ls="--", lw=1)
ax.text(CONFIG["HOURS_MAX"], ax.get_ylim()[1] * 0.9, f"  batas {CONFIG['HOURS_MAX']:,} jam", fontsize=8, color=C["ink2"])
ax.set_xlabel("Jam operasi (skala log, nilai > 0)")
ax.set_title("Distribusi hour meter")
subtitle(ax, f"Tambahan {(hrs == 0).sum():,} nilai nol dan {hrs.isna().sum():,} kosong tidak tampak di grafik")
plt.tight_layout()
plt.show()

#### (b) Korupsi data akibat Excel & ketidakseragaman format

In [ ]:
txt_cols = [c for c in train_raw.columns if train_raw[c].dtype == object or str(train_raw[c].dtype) == "str"]
corrupt = []
for c in txt_cols:
    s = train_raw[c].dropna().astype("string")
    for kind, mask in (("#NAME? (teks diawali '-' dibaca sbg rumus)", s.eq("#NAME?")),
                       ("Notasi ilmiah (teks seperti '214E4' dibaca sbg angka)", s.str.match(EXCEL_SCI))):
        if mask.sum():
            corrupt.append({"kolom": c, "jenis": kind, "baris": int(mask.sum()),
                            "contoh": ", ".join(s[mask].unique()[:3])})
display(pd.DataFrame(corrupt))

fmt = pd.DataFrame([
    ("Tire_Size", "angka vs angka+tanda inci", "17.5 / 17.5\""),
    ("Enclosure", "sinonim", "EROPS AC / EROPS w AC"),
    ("Stick_Length", "teks kaki-inci", "9' 6\" → perlu angka (inci)"),
    ("Undercarriage_Pad_Width", "teks dengan satuan", "32 inch → 32"),
    ("Blade_Width", "teks + simbol", "14' / <12'"),
    ("fiModelDesc", "tanda baca/spasi", "D155A-1 / D155A1; 580SUPER LII / 580SUPERLII"),
], columns=["Kolom", "Masalah", "Contoh"])
display(fmt)
print("Contoh variasi nilai Tire_Size:", train_raw["Tire_Size"].value_counts().head(12).index.tolist())

#### (c) Identitas mesin & model yang tidak andal

In [ ]:
g = train_raw.groupby("MachineID").agg(n_model=("ModelID", "nunique"), n_group=("ProductGroup", "nunique"),
                                       n_sales=("SalesID", "size"))
dm = train_raw.groupby("fiModelDesc").agg(n_model=("ModelID", "nunique"), n_class=("fiProductClassDesc", "nunique"))
dup_same_class = train_raw.groupby(["fiModelDesc", "fiProductClassDesc"])["ModelID"].nunique()
ident = pd.DataFrame([
    ("MachineID dengan > 1 ModelID", (g.n_model > 1).sum(), f"dari {len(g):,} MachineID"),
    ("MachineID dengan > 1 grup produk (mis. loader & excavator)", (g.n_group > 1).sum(), "→ ID dipakai ulang untuk alat berbeda"),
    ("ModelID dengan > 1 deskripsi", (train_raw.groupby("ModelID")["fiModelDesc"].nunique() > 1).sum(), "aturan 1 ModelID = 1 deskripsi"),
    ("Deskripsi yang dipakai > 1 ModelID", (dm.n_model > 1).sum(), "sebagian beda kelas produk (merek berbeda)"),
    ("Deskripsi + kelas sama tetapi > 1 ModelID", (dup_same_class > 1).sum(), "→ kandidat duplikat master model"),
], columns=["Pemeriksaan", "Jumlah", "Catatan"])
display(ident)

ex_id = g[(g.n_group > 1)].sort_values("n_sales", ascending=False).index[0]
print(f"Contoh MachineID {ex_id} — satu ID, beberapa alat berbeda:")
train_raw.loc[train_raw["MachineID"] == ex_id, ["SalesID", "saledate", "ModelID", "fiModelDesc", "ProductGroupDesc",
                                                "YearMade", "datasource"]].sort_values("saledate")

**Ringkasan Data Understanding**
* Kekosongan kolom spesifikasi sebagian besar **struktural (N/A)**; masalah kelengkapan sebenarnya adalah
  `None or Unspecified` pada atribut yang berlaku, `YearMade = 1000`, dan hour meter kosong/nol.
* **Validitas**: placeholder tahun, hour meter mustahil (hingga 2,48 juta jam), korupsi Excel (`#NAME?`, `2.14E+06`),
  format satuan tidak seragam.
* **Konsistensi & keunikan**: `MachineID` dipakai ulang untuk alat berbeda; mesin yang sama tercatat dengan tahun
  berbeda; model yang sama punya beberapa `ModelID`.
* Implikasi: perlu (1) aturan kualitas data yang eksplisit, (2) standardisasi, (3) **entity resolution** sebelum
  golden record dapat dibentuk.

## 3. Data Preparation — membangun master data
### 3.1 Katalog aturan kualitas data (*data quality rules*)
Aturan ditulis **deklaratif** (ID, dimensi, level entitas, deskripsi) dan dievaluasi oleh satu *rule engine* yang
sama untuk data **sebelum** dan **sesudah** pembersihan. Setiap aturan mengembalikan jumlah objek yang diperiksa
dan yang gagal, sehingga skor dapat dibandingkan secara adil.

In [ ]:
def norm_key(s: pd.Series) -> pd.Series:
    """Kunci pencocokan: huruf besar, hanya alfanumerik (D155A-1 == D155A1)."""
    return s.astype("string").str.upper().str.replace(r"[^A-Z0-9]", "", regex=True)


def known(s: pd.Series) -> pd.Series:
    return s.notna() & ~s.astype("string").isin(["Unknown", "N/A"] + list(SENTINELS)).fillna(False)


FORMAT_RULES = {   # kolom -> regex format terstandar
    "Enclosure": r"OROPS|EROPS|EROPS w AC|NO ROPS",
    "Tire_Size": r"\d+(\.\d+)?",
    "Undercarriage_Pad_Width": r"\d+(\.\d+)?( inch)?",
    "Stick_Length": r"\d+' \d+\"",
    "Blade_Width": r"<?\d+'",
}
MODEL_TEXT = ["fiModelDesc", "fiSecondaryDesc", "fiModelSeries", "fiModelDescriptor"]


def rule_catalog():
    """Setiap aturan: fungsi(view) -> (jumlah diperiksa, jumlah gagal).
    view = dict(sales=DataFrame transaksi, mkey=kolom kunci mesin, model=kolom kunci model)."""
    def r01(v):
        d = v["sales"]; y = d["YearMade"].dropna()
        return len(y), int((~y.between(CONFIG["YEAR_MIN"], d.loc[y.index, "saledate"].dt.year)).sum())

    def r02(v):
        d = v["sales"]; h = d["MachineHoursCurrentMeter"].dropna()
        yr = d.loc[h.index, "YearMade"].where(d.loc[h.index, "YearMade"].between(CONFIG["YEAR_MIN"], 2012))
        age = (d.loc[h.index, "saledate"].dt.year - yr).clip(lower=1)
        bad = ~h.between(1, CONFIG["HOURS_MAX"]) | ((h / age) > CONFIG["HOURS_PER_YEAR_MAX"]).fillna(False)
        return len(h), int(bad.sum())

    def r03(v):
        d = v["sales"].dropna(subset=["MachineHoursCurrentMeter"])
        d = d[d["MachineHoursCurrentMeter"] > 0].sort_values([v["mkey"], "saledate"])
        diff = d.groupby(v["mkey"])["MachineHoursCurrentMeter"].diff().dropna()
        return len(diff), int((diff < 0).sum())

    def r04(v):
        n = v["sales"].groupby(v["model"])["fiModelDesc"].nunique()
        return len(n), int((n > 1).sum())

    def r05(v):
        d = v["sales"]
        n = d.groupby([norm_key(d["fiModelDesc"]), d["fiProductClassDesc"]])[v["model"]].nunique()
        return len(n), int((n > 1).sum())

    def r06(v):
        n = v["sales"].groupby(v["mkey"])[v["model"]].nunique()
        return len(n), int((n > 1).sum())

    def r07(v):
        d = v["sales"]; d = d[d["YearMade"].between(CONFIG["YEAR_MIN"], 2012)]
        n = d.groupby(v["mkey"])["YearMade"].nunique()
        return len(n), int((n > 1).sum())

    def r08(v):
        d = v["sales"]
        n = (d.assign(_k=d["ProductGroup"].astype(str) + "|" + norm_key(d["fiBaseModel"]).fillna(""))
               .groupby(v["mkey"])["_k"].nunique())
        return len(n), int((n > 1).sum())

    def r09(v):
        vals = pd.concat([v["sales"][c].dropna().astype("string") for c in MODEL_TEXT])
        bad = vals.eq("#NAME?") | vals.str.match(EXCEL_SCI).fillna(False)
        return len(vals), int(bad.sum())

    def r10(v):
        checked = failed = 0
        for c, pat in FORMAT_RULES.items():
            s = v["sales"][c]
            s = s[known(s)].astype("string")
            checked += len(s); failed += int((~s.str.fullmatch(pat).fillna(False)).sum())
        return checked, failed

    def r11(v):
        d = v["sales"]
        comp = d[["fiBaseModel", "fiSecondaryDesc", "fiModelSeries", "fiModelDescriptor"]].fillna("").astype(str).agg("".join, axis=1)
        return len(d), int((norm_key(comp) != norm_key(d["fiModelDesc"])).fillna(True).sum())

    def r12(v):
        d = v["sales"]
        prefix = d["fiProductClassDesc"].astype("string").str.split(" - ").str[0]
        expected = d["ProductGroup"].map({"BL": "Backhoe Loader", "MG": "Motorgrader", "SSL": "Skid Steer Loader",
                                          "TEX": "Hydraulic Excavator, Track", "TTT": "Track Type Tractor, Dozer",
                                          "WL": "Wheel Loader"})
        return len(d), int((prefix != expected).fillna(True).sum())

    def r13(v):
        d = v["sales"].drop_duplicates(v["mkey"]) if v.get("entity_level") else v["sales"]
        checked = failed = 0
        for c in SPEC_COLS:
            app = applicable_mask(d, c)
            checked += int(app.sum()); failed += int((app & ~known(d[c])).sum())
        return checked, failed

    def r14(v):
        d = v["sales"]
        ok = d.assign(_ok=d["YearMade"].between(CONFIG["YEAR_MIN"], d["saledate"].dt.year)).groupby(v["mkey"])["_ok"].any()
        return len(ok), int((~ok).sum())

    def r15(v):
        s = v["sales"]["SalesID"]
        return len(s), int(s.duplicated().sum())

    return [
        ("R01", "Validitas", "transaksi", "Tahun pembuatan di antara 1950 dan tahun penjualan", r01),
        ("R02", "Validitas", "transaksi", "Hour meter 1–100.000 jam dan ≤ 8.760 jam per tahun umur", r02),
        ("R03", "Konsistensi", "pasangan transaksi", "Hour meter tidak turun untuk mesin yang sama", r03),
        ("R04", "Konsistensi", "model", "Satu ModelID → satu deskripsi model", r04),
        ("R05", "Keunikan", "model", "Satu model fisik (deskripsi + kelas) → satu ModelID", r05),
        ("R06", "Konsistensi", "mesin", "Satu mesin → satu model", r06),
        ("R07", "Konsistensi", "mesin", "Satu mesin → satu tahun pembuatan", r07),
        ("R08", "Keunikan", "mesin", "Satu ID mesin → satu alat fisik (grup & base model sama)", r08),
        ("R09", "Validitas", "sel teks", "Tidak ada korupsi Excel (#NAME?, notasi ilmiah)", r09),
        ("R10", "Validitas", "sel", "Atribut berformat standar (Enclosure, ban, track, stick, blade)", r10),
        ("R11", "Konsistensi", "transaksi", "Deskripsi model = gabungan komponennya", r11),
        ("R12", "Konsistensi", "transaksi", "Grup produk sesuai kelas produk", r12),
        ("R13", "Kelengkapan", "sel", "Atribut spesifikasi yang berlaku bernilai diketahui", r13),
        ("R14", "Kelengkapan", "mesin", "Tahun pembuatan valid diketahui", r14),
        ("R15", "Keunikan", "transaksi", "SalesID unik", r15),
    ]


RULES = rule_catalog()
RULE_SHORT = {"R01": "Tahun buat valid", "R02": "Hour meter masuk akal", "R03": "Hour meter tidak turun",
              "R04": "1 ModelID = 1 deskripsi", "R05": "1 model fisik = 1 ModelID", "R06": "1 mesin = 1 model",
              "R07": "1 mesin = 1 tahun buat", "R08": "1 ID = 1 alat fisik", "R09": "Bebas korupsi Excel",
              "R10": "Format atribut standar", "R11": "Deskripsi = gabungan komponen",
              "R12": "Grup sesuai kelas produk", "R13": "Spesifikasi berlaku diketahui",
              "R14": "Tahun buat diketahui", "R15": "SalesID unik"}


def run_rules(view, stage):
    rows = []
    for rid, dim, level, desc, fn in RULES:
        checked, failed = fn(view)
        rows.append({"rule_id": rid, "dimensi": dim, "level": level, "aturan": desc, "tahap": stage,
                     "diperiksa": checked, "gagal": failed, "pass_rate": 1 - failed / checked if checked else 1.0})
    return pd.DataFrame(rows)


raw_view = {"sales": train_raw, "mkey": "MachineID", "model": "ModelID"}
rules_before = run_rules(raw_view, "sebelum")
rules_before[["rule_id", "dimensi", "aturan", "diperiksa", "gagal", "pass_rate"]]

In [ ]:
by_src = []
for src, d in train_raw.groupby("datasource"):
    r = run_rules({"sales": d, "mkey": "MachineID", "model": "ModelID"}, f"source {src}")
    by_src.append(r.set_index("rule_id")["pass_rate"].rename(f"datasource {src}\n(n={len(d):,})"))
by_src = pd.concat(by_src, axis=1)
fig, ax = plt.subplots(figsize=(10, 6.5))
im = ax.imshow(by_src.values, cmap=rg, vmin=0.5, vmax=1, aspect="auto")
ax.set_xticks(range(by_src.shape[1]), by_src.columns, fontsize=8.5)
ax.set_yticks(range(len(by_src)), [f"{r} {RULE_SHORT[r]}" for r in by_src.index], fontsize=8.5)
for i in range(by_src.shape[0]):
    for j in range(by_src.shape[1]):
        ax.text(j, i, f"{by_src.iat[i, j]:.0%}", ha="center", va="center", fontsize=8, color=C["ink"])
ax.grid(visible=False)
ax.set_title("Pass rate aturan kualitas data per sumber data (sebelum pembersihan)")
w2, w3 = by_src.loc["R02"].idxmin().split("\n")[0], by_src.loc["R03"].idxmin().split("\n")[0]
subtitle(ax, f"Hour meter paling bermasalah di {w2} (R02 {by_src.loc['R02'].min():.0%}) dan {w3} "
             f"(R03 {by_src.loc['R03'].min():.0%}) → perbaikan perlu di proses sumber")
plt.tight_layout()
plt.show()

### 3.2 Standardisasi
1. **Teks**: trim & rapikan spasi.
2. **Perbaikan korupsi Excel**: deskripsi `2.14E+06` direkonstruksi dari komponennya (`214`+`E`+`4` → `214E4`);
   komponen `#NAME?`/notasi ilmiah diturunkan dari sisa deskripsi (`SK160LC-IV` − `SK160` − `LC` → `-IV`).
3. **Semantik nilai kosong**: `None or Unspecified` atau kosong pada atribut yang **berlaku** → `Unknown`;
   kosong pada atribut yang **tidak berlaku** → `N/A`. Keduanya tidak dicampur (berbeda makna bisnis).
4. **Domain & satuan**: sinonim (`EROPS AC` → `EROPS w AC`), satuan seragam + kolom numerik
   (`17.5"` → 17,5 in; `9' 6"` → 114 in; `32 inch` → 32 in; `14'` → 14 ft).
5. **Validitas**: tahun di luar 1950–tahun jual dan hour meter tidak masuk akal → kosong + *flag* (nilai asli tetap disimpan).
6. **Kelas produk** diurai menjadi tipe, batas bawah, batas atas, dan satuan (mis. 110–120 Horsepower).

In [ ]:
def _residual(desc, *parts):
    """Sisa deskripsi setelah membuang prefix komponen yang diketahui."""
    out = desc
    for p in parts:
        if isinstance(p, str) and p and out.upper().startswith(p.upper()):
            out = out[len(p):]
    return out


def repair_excel(df):
    d = df.copy()
    log = {}
    is_bad = lambda s: s.astype("string").eq("#NAME?").fillna(False) | s.astype("string").str.match(EXCEL_SCI).fillna(False)
    # (a) deskripsi rusak -> rekonstruksi dari komponen
    bad = is_bad(d["fiModelDesc"])
    comps = d.loc[bad, ["fiBaseModel", "fiSecondaryDesc", "fiModelSeries", "fiModelDescriptor"]].fillna("").astype(str)
    d.loc[bad, "fiModelDesc"] = comps.agg("".join, axis=1)
    log["fiModelDesc direkonstruksi"] = int(bad.sum())
    # (b) komponen rusak -> turunkan dari sisa deskripsi
    for col, before, after in (("fiSecondaryDesc", ["fiBaseModel"], ["fiModelSeries", "fiModelDescriptor"]),
                               ("fiModelSeries", ["fiBaseModel", "fiSecondaryDesc"], ["fiModelDescriptor"]),
                               ("fiModelDescriptor", ["fiBaseModel", "fiSecondaryDesc", "fiModelSeries"], [])):
        bad = is_bad(d[col])
        fixed = 0
        for i in d.index[bad]:
            res = _residual(d.at[i, "fiModelDesc"], *[d.at[i, c] for c in before])
            for c in after:   # buang suffix komponen sesudahnya bila valid
                sfx = d.at[i, c]
                if isinstance(sfx, str) and sfx and not is_bad(pd.Series([sfx])).iat[0] and res.upper().endswith(sfx.upper()):
                    res = res[: -len(sfx)]
            d.at[i, col] = res if res else pd.NA
            fixed += bool(res)
        log[f"{col} diperbaiki"] = fixed
    return d, log


def to_inches_stick(s):
    m = s.astype("string").str.extract(r"(\d+)' (\d+)\"")
    return m[0].astype(float) * 12 + m[1].astype(float)


def standardize(df: pd.DataFrame):
    d = df.copy()
    text_cols = [c for c in d.columns if d[c].dtype == object or str(d[c].dtype) in ("str", "string")]
    for c in text_cols:
        d[c] = d[c].astype("string").str.strip().str.replace(r"\s+", " ", regex=True)
    d, repair_log = repair_excel(d)

    # domain & satuan
    d["Enclosure"] = d["Enclosure"].replace({"EROPS AC": "EROPS w AC"})
    d["Tire_Size"] = d["Tire_Size"].str.replace('"', "", regex=False).str.replace(" inch", "", regex=False)
    d["tire_size_in"] = pd.to_numeric(d["Tire_Size"], errors="coerce")
    d["pad_width_in"] = pd.to_numeric(d["Undercarriage_Pad_Width"].str.replace(" inch", "", regex=False), errors="coerce")
    d["stick_length_in"] = to_inches_stick(d["Stick_Length"])
    d["blade_width_ft"] = pd.to_numeric(d["Blade_Width"].str.extract(r"^(\d+)'$")[0], errors="coerce")

    # semantik kosong: Unknown (berlaku) vs N/A (tidak berlaku)
    for c in SPEC_COLS:
        app = applicable_mask(d, c)
        unk = d[c].isna() | d[c].isin(list(SENTINELS)).fillna(False)
        d[c] = d[c].mask(unk & app, "Unknown").mask(unk & ~app, "N/A")
    d["ProductSize"] = d["ProductSize"].fillna("Unknown")

    # validitas tahun & hour meter (nilai asli disimpan untuk lineage)
    d["year_made_raw"] = d["YearMade"]
    valid_year = d["YearMade"].between(CONFIG["YEAR_MIN"], d["saledate"].dt.year)
    d["year_flag"] = np.where(d["YearMade"].eq(1000), "placeholder", np.where(valid_year, "valid", "invalid"))
    d["YearMade"] = d["YearMade"].where(valid_year)
    h = d["MachineHoursCurrentMeter"]
    d["hours_raw"] = h
    age = (d["saledate"].dt.year - d["YearMade"]).clip(lower=1)
    plausible = h.between(1, CONFIG["HOURS_MAX"]) & ~((h / age) > CONFIG["HOURS_PER_YEAR_MAX"]).fillna(False)
    d["hours_flag"] = np.select([h.isna(), h.eq(0), ~plausible], ["missing", "zero_placeholder", "implausible"], "valid")
    d["MachineHoursCurrentMeter"] = h.where(plausible)

    # kelas produk terurai
    pc = d["fiProductClassDesc"].str.extract(r"^(.*?) - ([\d.]+) to ([\d.]+) (.*)$")
    d["class_type"], d["class_unit"] = pc[0], pc[3]
    d["class_low"], d["class_high"] = pc[1].astype(float), pc[2].astype(float)

    # kunci pencocokan
    d["desc_norm"] = norm_key(d["fiModelDesc"])
    d["base_norm"] = norm_key(d["fiBaseModel"])
    return d, repair_log


std_train, repair_log = standardize(train_raw)
display(pd.Series(repair_log, name="jumlah baris").to_frame())
print("Contoh hasil perbaikan Excel:")
mask_fix = train_raw["fiModelDesc"].astype("string").str.match(EXCEL_SCI).fillna(False) | train_raw["fiModelSeries"].eq("#NAME?")
pd.concat([train_raw.loc[mask_fix, ["fiModelDesc", "fiModelSeries"]].add_suffix(" (asli)"),
           std_train.loc[mask_fix, ["fiModelDesc", "fiModelSeries"]].add_suffix(" (standar)")], axis=1).drop_duplicates().head(8)

### 3.3 Master model (deduplikasi `ModelID`)
Model fisik yang sama diidentifikasi dengan **kunci pencocokan = deskripsi ternormalisasi + kelas produk**
(deskripsi saja tidak cukup: `120` dipakai motor grader dan excavator dari merek berbeda). Dari setiap kelompok
duplikat dipilih **satu ModelID kanonik** (paling banyak transaksi), dan semua ModelID sumber dipetakan lewat
tabel *crosswalk* `xref_model`.

In [ ]:
def mode_by(df, key, col, tiebreak="saledate"):
    """Nilai paling sering per entitas (tie-break: transaksi terbaru) + rasio kesepakatan (agreement)."""
    d = df.loc[known(df[col]) if df[col].dtype != float else df[col].notna(), [key, col, tiebreak]]
    g = d.groupby([key, col], observed=True).agg(n=(tiebreak, "size"), last=(tiebreak, "max")).reset_index()
    g = g.sort_values([key, "n", "last"], ascending=[True, False, False])
    g["agreement"] = g["n"] / g.groupby(key)["n"].transform("sum")
    return g.drop_duplicates(key).set_index(key)[[col, "agreement"]]


def build_model_master(std):
    key = std["desc_norm"] + "|" + std["fiProductClassDesc"]
    counts = std.groupby([key.rename("match_key"), "ModelID"]).size().rename("n").reset_index()
    counts = counts.sort_values(["match_key", "n", "ModelID"], ascending=[True, False, True])
    canon = counts.drop_duplicates("match_key").set_index("match_key")["ModelID"]
    xref = (std.assign(match_key=key).groupby("ModelID")
              .agg(match_key=("match_key", lambda s: s.mode().iat[0]), source_desc=("fiModelDesc", "first"),
                   n_sales=("SalesID", "size")).reset_index())
    xref["model_key"] = xref["match_key"].map(canon).astype(int)
    s = std.assign(model_key=std["ModelID"].map(xref.set_index("ModelID")["model_key"]))
    first = s.groupby("model_key").agg(product_group=("ProductGroup", "first"),
                                       product_group_desc=("ProductGroupDesc", "first"),
                                       product_class=("fiProductClassDesc", "first"),
                                       class_type=("class_type", "first"), class_low=("class_low", "first"),
                                       class_high=("class_high", "first"), class_unit=("class_unit", "first"),
                                       n_sales=("SalesID", "size"), n_source_ids=("ModelID", "nunique"))
    # komponen deskripsi dipilih SEBAGAI SATU KOMBINASI (bukan per kolom) agar tetap konsisten satu sama lain
    comp_cols = ["fiModelDesc", "fiBaseModel", "fiSecondaryDesc", "fiModelSeries", "fiModelDescriptor"]
    s["_tuple"] = s[comp_cols].fillna("").astype(str).agg("\x1f".join, axis=1)
    tup = mode_by(s, "model_key", "_tuple")["_tuple"].str.split("\x1f", expand=True)
    tup.columns = comp_cols
    tup = tup.replace("", pd.NA)
    attrs = tup.join(mode_by(s, "model_key", "ProductSize")["ProductSize"])
    dim = first.join(attrs).rename(columns={
        "fiModelDesc": "model_desc", "fiBaseModel": "base_model", "fiSecondaryDesc": "secondary_desc",
        "fiModelSeries": "model_series", "fiModelDescriptor": "model_descriptor", "ProductSize": "product_size"})
    dim["product_size"] = dim["product_size"].fillna("Unknown")
    return dim.reset_index(), xref.rename(columns={"ModelID": "source_model_id"})


dim_model, xref_model = build_model_master(std_train)
merged = xref_model[xref_model["source_model_id"] != xref_model["model_key"]]
print(f"ModelID sumber: {len(xref_model):,} → model master: {len(dim_model):,} "
      f"({len(merged):,} ModelID duplikat digabung)")
print("Contoh kelompok duplikat yang digabung:")
ex_keys = merged["model_key"].value_counts().index[:4]
xref_model[xref_model["model_key"].isin(ex_keys)].sort_values(["model_key", "n_sales"], ascending=[True, False])

### 3.4 *Entity resolution* & golden record mesin
`MachineID` sumber tidak dapat langsung dipakai sebagai identitas (Bagian 2.4c). Aturan pencocokan:

| Aturan | Logika | Alasan |
|---|---|---|
| ER1 | Grup produk berbeda → mesin berbeda | satu unit tidak mungkin sekaligus loader dan excavator |
| ER2 | Base model ternormalisasi berbeda → mesin berbeda | WA500 ≠ PC300 |
| ER3 | Tahun pembuatan valid berbeda > 1 tahun → mesin berbeda | selisih 1 tahun lazim (model year vs tahun produksi / salah ketik) |
| ER4 | Transaksi tanpa tahun valid → digabung ke klaster terbesar pada kunci yang sama (*confidence* rendah bila ada > 1 klaster) | tidak ada bukti untuk memisahkan |

**Survivorship** (aturan pemilihan nilai golden):
* Model → model kanonik paling sering (tie: transaksi terbaru); tahun → nilai valid paling sering (tie: terbaru).
* Spesifikasi → nilai **diketahui** paling sering dari transaksi mesin itu sendiri; bila tidak ada → `Unknown`/`N/A`
  (tidak diwarisi diam-diam dari model — pewarisan hanya dipakai sebagai fitur ML dan ditandai).
* Hour meter → bacaan valid terakhir; bacaan yang **turun** dibanding bacaan sebelumnya ditandai `decreasing` dan dikarantina.
* Skor *confidence* = rata-rata agreement model & tahun × penalti ER4.

In [ ]:
def resolve_entities(std):
    d = std[["SalesID", "MachineID", "ProductGroup", "base_norm", "YearMade", "saledate"]].copy()
    d["k0"] = d["MachineID"].astype(str) + "|" + d["ProductGroup"].astype(str) + "|" + d["base_norm"].fillna("")
    # ER3: klaster tahun dengan lebar maksimum 1 tahun
    years = d.dropna(subset=["YearMade"])[["k0", "YearMade"]].drop_duplicates().sort_values(["k0", "YearMade"])
    start, cid, cur_k, cur_start = [], [], None, None
    counter = 0
    for k, y in zip(years["k0"].to_numpy(), years["YearMade"].to_numpy()):
        if k != cur_k:
            cur_k, cur_start, counter = k, y, 0
        elif y - cur_start > 1:
            cur_start, counter = y, counter + 1
        cid.append(counter)
    years["year_cluster"] = cid
    d = d.merge(years, on=["k0", "YearMade"], how="left")
    # ER4: tanpa tahun -> klaster terbesar
    size = d.dropna(subset=["year_cluster"]).groupby(["k0", "year_cluster"]).size().rename("n").reset_index()
    largest = size.sort_values(["k0", "n", "year_cluster"], ascending=[True, False, True]).drop_duplicates("k0")
    n_clusters = size.groupby("k0").size()
    d["year_cluster"] = d["year_cluster"].fillna(d["k0"].map(largest.set_index("k0")["year_cluster"])).fillna(0).astype(int)
    d["er_rule"] = np.where(d["YearMade"].isna() & d["k0"].map(n_clusters).fillna(1).gt(1), "ER4-low-confidence", "ER1-3")
    # nomor urut entitas per MachineID (urut transaksi pertama)
    ent = d.groupby(["MachineID", "k0", "year_cluster"])["saledate"].min().rename("first").reset_index()
    ent = ent.sort_values(["MachineID", "first", "k0", "year_cluster"])
    ent["seq"] = ent.groupby("MachineID").cumcount() + 1
    ent["machine_key"] = ent["MachineID"].astype(str) + "-" + ent["seq"].astype(str)
    d = d.merge(ent[["MachineID", "k0", "year_cluster", "machine_key"]], on=["MachineID", "k0", "year_cluster"], how="left")
    return d.set_index("SalesID")[["machine_key", "er_rule"]]


def build_golden(std, xref_model, dim_model):
    er = resolve_entities(std)
    s = std.set_index("SalesID").join(er).reset_index()
    s["model_key"] = s["ModelID"].map(xref_model.set_index("source_model_id")["model_key"])
    # hour meter monoton: bacaan yang turun dikarantina
    s = s.sort_values(["machine_key", "saledate", "SalesID"])
    prev = s.groupby("machine_key")["MachineHoursCurrentMeter"].shift()
    prev_max = prev.groupby(s["machine_key"]).cummax().groupby(s["machine_key"]).ffill()
    dec = s["MachineHoursCurrentMeter"] < prev_max
    s.loc[dec, "hours_flag"] = "decreasing"
    s.loc[dec, "MachineHoursCurrentMeter"] = np.nan

    g = s.groupby("machine_key").agg(source_machine_id=("MachineID", "first"), n_sales=("SalesID", "size"),
                                     first_sale_date=("saledate", "min"), last_sale_date=("saledate", "max"),
                                     max_hours=("MachineHoursCurrentMeter", "max"),
                                     low_conf=("er_rule", lambda x: (x == "ER4-low-confidence").any()))
    mk = mode_by(s, "machine_key", "model_key").rename(columns={"agreement": "model_agreement"})
    yr = mode_by(s, "machine_key", "YearMade").rename(columns={"YearMade": "year_made", "agreement": "year_agreement"})
    last_state = s.sort_values("saledate").groupby("machine_key")["state"].last().rename("last_state")
    golden = g.join(mk).join(yr).join(last_state)
    golden["model_key"] = golden["model_key"].astype(int)
    golden["year_made"] = golden["year_made"].round().astype("Int64")
    golden = golden.join(dim_model.set_index("model_key")[["product_group", "model_desc"]], on="model_key")
    for c in SPEC_COLS + ["tire_size_in", "pad_width_in", "stick_length_in", "blade_width_ft"]:
        golden[c] = mode_by(s, "machine_key", c)[c]
    for c in SPEC_COLS:   # tidak diketahui -> Unknown / N/A sesuai applicability
        app = golden["product_group"].map(APPLICABLE.loc[c]).fillna(False).astype(bool)
        golden[c] = golden[c].where(golden[c].notna(), np.where(app, "Unknown", "N/A"))
    golden["confidence"] = (golden["model_agreement"].fillna(1) * golden["year_agreement"].fillna(0.5)
                            * np.where(golden["low_conf"], 0.8, 1.0)).round(3)
    golden = golden.drop(columns=["low_conf"]).reset_index()
    xref_machine = (s.groupby(["MachineID", "machine_key"]).agg(n_sales=("SalesID", "size"),
                                                                er_rule=("er_rule", "first"))
                     .reset_index().rename(columns={"MachineID": "source_machine_id"}))
    return golden, xref_machine, s


t = time.time()
dim_machine, xref_machine, fact_train = build_golden(std_train, xref_model, dim_model)
split_ids = xref_machine.groupby("source_machine_id").size()
print(f"Selesai dalam {time.time() - t:.0f} detik")
pd.DataFrame([
    ("MachineID sumber", f"{train_raw['MachineID'].nunique():,}"),
    ("Golden record mesin", f"{len(dim_machine):,}"),
    ("MachineID dipecah menjadi > 1 mesin", f"{(split_ids > 1).sum():,}"),
    ("Mesin dengan confidence < 0,8", f"{(dim_machine['confidence'] < 0.8).sum():,}"),
    ("Bacaan hour meter dikarantina (turun)", f"{(fact_train['hours_flag'] == 'decreasing').sum():,}"),
], columns=["Ukuran", "Nilai"])

In [ ]:
ex = split_ids.sort_values(ascending=False).index[0]
print(f"Hasil entity resolution untuk MachineID {ex}:")
display(dim_machine.loc[dim_machine["source_machine_id"] == ex,
                        ["machine_key", "model_desc", "product_group", "year_made", "n_sales", "first_sale_date",
                         "last_sale_date", "confidence"]].sort_values("first_sale_date"))

fig, axes = plt.subplots(1, 2, figsize=(14, 4.2))
ax = axes[0]
vc = split_ids.value_counts().sort_index()
vc.index = [str(i) if i < 6 else "6+" for i in vc.index]
vc = vc.groupby(level=0, sort=False).sum()
ax.bar(vc.index, vc.values, color=[C["blue"]] + [C["orange"]] * (len(vc) - 1), width=0.6)
ax.set_yscale("log")
for i, v in enumerate(vc.values):
    ax.text(i, v, f"{v:,}", ha="center", va="bottom", fontsize=8.5)
ax.set_xlabel("Jumlah golden record per MachineID sumber")
ax.set_ylabel("Jumlah MachineID (log)")
ax.set_title("Hasil entity resolution")
subtitle(ax, f"{(split_ids > 1).sum():,} MachineID ternyata mewakili > 1 alat fisik (oranye)")
ax.grid(axis="x", visible=False)
ax = axes[1]
cat_conf = pd.Series(np.select(
    [dim_machine["year_made"].isna(),
     (dim_machine["model_agreement"] < 1) | (dim_machine["year_agreement"] < 1),
     dim_machine["confidence"] < 1],
    ["Tahun pembuatan tidak diketahui", "Ada konflik model/tahun antar transaksi", "Penugasan ER4 (tahun tak diketahui)"],
    "Konsisten penuh (confidence 1,0)")).value_counts()
order = ["Konsisten penuh (confidence 1,0)", "Ada konflik model/tahun antar transaksi",
         "Penugasan ER4 (tahun tak diketahui)", "Tahun pembuatan tidak diketahui"]
cat_conf = cat_conf.reindex(order).fillna(0)
colors = [C["good"], C["warn"], C["warn"], C["muted"]]
ax.barh(cat_conf.index[::-1], cat_conf.values[::-1], color=colors[::-1], height=0.55)
for i, v in enumerate(cat_conf.values[::-1]):
    ax.text(v, i, f"  {v:,.0f} ({v / len(dim_machine):.1%})", va="center", fontsize=9)
ax.set_xlim(0, cat_conf.max() * 1.35)
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v / 1000:,.0f}k"))
ax.set_xlabel("Jumlah golden record mesin")
ax.set_title("Kondisi golden record")
subtitle(ax, "Kuning & abu = prioritas verifikasi manual (cek plat nomor seri)")
ax.grid(axis="y", visible=False)
plt.tight_layout()
plt.show()

### 3.5 Scorecard kualitas data: sebelum vs sesudah
Aturan yang sama dievaluasi pada data **sesudah** pembersihan: transaksi terstandar yang memakai kunci golden
(`machine_key`, `model_key`), tahun golden, dan hour meter yang telah dikarantina. Untuk kelengkapan spesifikasi,
yang dihitung hanya nilai **yang benar-benar berasal dari transaksi mesin itu sendiri** (survivorship),
bukan nilai warisan dari model — agar skor tidak "dipercantik" oleh imputasi.

In [ ]:
clean_sales = fact_train.copy()
gm = dim_machine.set_index("machine_key")
clean_sales["model_key"] = clean_sales["machine_key"].map(gm["model_key"])          # model golden
clean_sales["YearMade"] = clean_sales["machine_key"].map(gm["year_made"])           # tahun golden
mdl = dim_model.set_index("model_key")
for src, dst in (("model_desc", "fiModelDesc"), ("base_model", "fiBaseModel"), ("secondary_desc", "fiSecondaryDesc"),
                 ("model_series", "fiModelSeries"), ("model_descriptor", "fiModelDescriptor"),
                 ("product_class", "fiProductClassDesc"), ("product_group", "ProductGroup")):
    clean_sales[dst] = clean_sales["model_key"].map(mdl[src])
for c in SPEC_COLS:                                                                 # spesifikasi golden
    clean_sales[c] = clean_sales["machine_key"].map(gm[c])
clean_view = {"sales": clean_sales, "mkey": "machine_key", "model": "model_key"}
rules_after = run_rules(clean_view, "sesudah")

cmp = rules_before.merge(rules_after[["rule_id", "diperiksa", "gagal", "pass_rate"]], on="rule_id",
                         suffixes=("_sebelum", "_sesudah"))
cmp[["rule_id", "dimensi", "aturan", "gagal_sebelum", "pass_rate_sebelum", "gagal_sesudah", "pass_rate_sesudah"]]

In [ ]:
def scorecard(rules):
    dim = rules.groupby("dimensi")["pass_rate"].mean()
    return pd.concat([dim, pd.Series({"KESELURUHAN": dim.mean()})])


score = pd.DataFrame({"sebelum": scorecard(rules_before), "sesudah": scorecard(rules_after)})
score["perubahan (poin)"] = (score["sesudah"] - score["sebelum"]) * 100
display(score)

fig, axes = plt.subplots(1, 2, figsize=(15, 5.2), gridspec_kw={"width_ratios": [1, 1.5]})
ax = axes[0]
sc = score.drop("KESELURUHAN")
yy = np.arange(len(sc))
ax.barh(yy + 0.2, sc["sebelum"], height=0.4, color=C["muted"], label="Sebelum")
ax.barh(yy - 0.2, sc["sesudah"], height=0.4, color=C["blue"], label="Sesudah")
for i, (b, a) in enumerate(zip(sc["sebelum"], sc["sesudah"])):
    ax.text(b + 0.005, i + 0.2, f"{b:.1%}", va="center", fontsize=8.5, color=C["ink2"])
    ax.text(a + 0.005, i - 0.2, f"{a:.1%}", va="center", fontsize=8.5, color=C["ink"])
ax.set_yticks(yy, sc.index)
ax.set_xlim(0, 1.15)
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.0%}"))
ax.legend(loc="lower right", fontsize=8.5)
ax.set_title("Data Quality Scorecard per dimensi")
subtitle(ax, f"Skor keseluruhan {score.loc['KESELURUHAN', 'sebelum']:.1%} → {score.loc['KESELURUHAN', 'sesudah']:.1%}")
ax.grid(axis="y", visible=False)
ax = axes[1]
cm = cmp.set_index("rule_id")
yy = np.arange(len(cm))[::-1]
for y_, (rid, r) in zip(yy, cm.iterrows()):
    ax.plot([r.pass_rate_sebelum, r.pass_rate_sesudah], [y_, y_], color=C["grid"], lw=3, zorder=1)
    ax.scatter(r.pass_rate_sebelum, y_, color=C["muted"], s=40, zorder=2)
    ax.scatter(r.pass_rate_sesudah, y_, color=C["blue"], s=40, zorder=3)
ax.set_yticks(yy, [f"{rid} {RULE_SHORT[rid]}" for rid in cm.index], fontsize=8.5)
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.0%}"))
ax.set_xlabel("Pass rate (abu = sebelum, biru = sesudah)")
ax.set_title("Pass rate per aturan")
subtitle(ax, "Kelengkapan (R13, R14) tidak naik: pembersihan tidak boleh mengarang data")
plt.tight_layout()
plt.show()

**Mengapa kelengkapan tidak naik?** Golden record hanya dapat mengisi nilai yang kosong bila transaksi lain dari
**mesin yang sama** memuat nilainya. Pemeriksaan di bawah menunjukkan hal itu praktis tidak pernah terjadi: bila
sebuah spesifikasi tidak diketahui, ia tidak diketahui di semua transaksi mesin tersebut. Artinya kelengkapan
hanya dapat diperbaiki **di sumber** (field wajib saat input data), bukan oleh pembersihan. Pewarisan nilai dari
master model dapat menaikkan cakupan, tetapi itu **imputasi** (asumsi "unit sama dengan konfigurasi paling umum
modelnya") sehingga tidak dihitung sebagai kelengkapan — hanya dipakai sebagai fitur ML yang ditandai.

In [ ]:
multi = fact_train[fact_train.duplicated("machine_key", keep=False)]
mixed = {c: int(multi.assign(k=known(multi[c])).groupby("machine_key")["k"].agg(lambda x: 0 < x.sum() < len(x)).sum())
         for c in ["Enclosure", "Hydraulics", "Tire_Size", "Track_Type", "Blade_Type", "Ride_Control"]}
print(f"Mesin dengan > 1 transaksi: {multi['machine_key'].nunique():,}")
print("Mesin yang spesifikasinya diketahui di sebagian transaksi saja (bisa dilengkapi golden):", mixed)

inherit = fact_train.copy()
for c in SPEC_COLS:
    mm = mode_by(inherit, "model_key", c)[c]
    inherit[c] = inherit[c].where(known(inherit[c]), inherit["model_key"].map(mm).fillna(inherit[c]))
pot = run_rules({"sales": inherit, "mkey": "machine_key", "model": "model_key"}, "x").set_index("rule_id").loc["R13", "pass_rate"]
print(f"Kelengkapan spesifikasi (R13): sumber = {rules_after.set_index('rule_id').loc['R13', 'pass_rate']:.1%} | "
      f"bila diwarisi dari master model (imputasi, TIDAK dihitung) = {pot:.1%}")

### 3.6 Integrasi batch data baru (Valid & Test 2012)
Master data bukan proyek sekali jalan. Batch baru melewati standardisasi yang sama, lalu dicocokkan ke golden
record: **cocok** (kunci & klaster tahun sama), **entitas baru pada MachineID lama** (tabrakan ID), atau **mesin baru**.

In [ ]:
std_new, _ = standardize(pd.concat([valid_raw, test_raw], ignore_index=True))
xref_new = xref_model.copy()
unknown_models = ~std_new["ModelID"].isin(xref_new["source_model_id"])
# ModelID baru: cocokkan via kunci deskripsi + kelas ke model master; bila tidak ada -> model baru
mk_lookup = (dim_model.assign(match_key=norm_key(dim_model["model_desc"]) + "|" + dim_model["product_class"])
             .drop_duplicates("match_key").set_index("match_key")["model_key"])
new_ids = std_new.loc[unknown_models].drop_duplicates("ModelID")
new_ids_key = (new_ids["desc_norm"] + "|" + new_ids["fiProductClassDesc"]).map(mk_lookup)
print(f"ModelID baru di batch 2012: {len(new_ids)} → cocok ke model master: {new_ids_key.notna().sum()}, "
      f"model baru: {new_ids_key.isna().sum()}")

combined = pd.concat([std_train.assign(_batch="historis"), std_new.assign(_batch="baru")], ignore_index=True)
dim_model_all, xref_model_all = build_model_master(combined)
dim_machine_all, xref_machine_all, fact_all = build_golden(combined, xref_model_all, dim_model_all)
hist_keys = set(fact_all.loc[fact_all["_batch"] == "historis", "machine_key"])
hist_ids = set(train_raw["MachineID"])
nb = fact_all[fact_all["_batch"] == "baru"]
match_status = np.select([nb["machine_key"].isin(hist_keys), nb["MachineID"].isin(hist_ids)],
                         ["Cocok dgn golden record lama", "Entitas baru pada MachineID lama"], "Mesin baru")
ms = pd.Series(match_status).value_counts()
display(ms.rename("transaksi").to_frame().assign(proporsi=lambda x: x["transaksi"] / x["transaksi"].sum()))

fig, ax = plt.subplots(figsize=(8, 3))
cols_ms = {"Cocok dgn golden record lama": C["blue"], "Entitas baru pada MachineID lama": C["orange"], "Mesin baru": C["muted"]}
left = 0
for k in cols_ms:
    v = ms.get(k, 0) / ms.sum()
    ax.barh([0], [v], left=left, color=cols_ms[k], height=0.5, label=k)
    if v > 0.04:
        ax.text(left + v / 2, 0, f"{v:.0%}", ha="center", va="center", color="white", fontsize=10, fontweight="bold")
    left += v
ax.set_xlim(0, 1); ax.set_yticks([])
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.0%}"))
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.25), ncol=3, fontsize=8.5)
ax.set_title(f"Pencocokan {len(nb):,} transaksi batch 2012 ke master")
subtitle(ax, "Oranye = tabrakan ID terdeteksi otomatis sehingga riwayat unit tidak tercampur")
ax.grid(visible=False)
plt.tight_layout()
plt.show()

### 3.7 Menyimpan ke database SQL (SQLite) — skema master & transaksi
```
dim_model (master model) 1 ── * xref_model (ModelID sumber → model kanonik)
    1
    └── * dim_machine (golden record mesin) 1 ── * xref_machine (MachineID sumber → golden)
                     1
                     └── * fact_sale (transaksi; harga NULL untuk batch 2012)
dq_rule_result, dq_scorecard  (hasil aturan & skor per tahap — sumber dashboard)
```

In [ ]:
DB_PATH = CONFIG["OUT_DIR"] / "heavy_equipment_master.db"
DB_PATH.unlink(missing_ok=True)
DDL = f"""
PRAGMA foreign_keys = ON;
CREATE TABLE dim_model (
    model_key INTEGER PRIMARY KEY, model_desc TEXT NOT NULL, base_model TEXT, secondary_desc TEXT,
    model_series TEXT, model_descriptor TEXT, product_group TEXT NOT NULL, product_group_desc TEXT,
    product_class TEXT NOT NULL, class_type TEXT, class_low REAL, class_high REAL, class_unit TEXT,
    product_size TEXT, n_source_ids INTEGER, n_sales INTEGER);
CREATE TABLE xref_model (
    source_model_id INTEGER PRIMARY KEY, model_key INTEGER NOT NULL REFERENCES dim_model(model_key),
    source_desc TEXT, match_key TEXT, n_sales INTEGER);
CREATE TABLE dim_machine (
    machine_key TEXT PRIMARY KEY, source_machine_id INTEGER NOT NULL,
    model_key INTEGER NOT NULL REFERENCES dim_model(model_key), year_made INTEGER,
    model_agreement REAL, year_agreement REAL, confidence REAL, n_sales INTEGER,
    first_sale_date TEXT, last_sale_date TEXT, last_state TEXT, max_hours REAL,
    tire_size_in REAL, pad_width_in REAL, stick_length_in REAL, blade_width_ft REAL);
CREATE TABLE machine_attribute (      -- spesifikasi golden, hanya nilai yang diketahui (format panjang)
    machine_key TEXT NOT NULL REFERENCES dim_machine(machine_key), attribute TEXT NOT NULL,
    value TEXT NOT NULL, PRIMARY KEY (machine_key, attribute));
CREATE TABLE xref_machine (
    source_machine_id INTEGER NOT NULL, machine_key TEXT NOT NULL REFERENCES dim_machine(machine_key),
    n_sales INTEGER, er_rule TEXT, PRIMARY KEY (source_machine_id, machine_key));
CREATE TABLE fact_sale (
    sales_id INTEGER PRIMARY KEY, machine_key TEXT NOT NULL REFERENCES dim_machine(machine_key),
    model_key_recorded INTEGER NOT NULL REFERENCES dim_model(model_key), source_model_id INTEGER,
    source_machine_id INTEGER, sale_date TEXT NOT NULL, sale_price REAL, hours_meter REAL, hours_raw REAL,
    hours_flag TEXT, year_made_raw INTEGER, year_flag TEXT, usage_band TEXT, state TEXT,
    auctioneer_id INTEGER, datasource INTEGER, source_batch TEXT, er_rule TEXT);
CREATE INDEX ix_fact_machine ON fact_sale(machine_key);
CREATE INDEX ix_fact_date ON fact_sale(sale_date);
CREATE INDEX ix_machine_model ON dim_machine(model_key);
CREATE TABLE dq_rule_result (
    rule_id TEXT, dimension TEXT, level TEXT, description TEXT, stage TEXT, checked INTEGER,
    failed INTEGER, pass_rate REAL, PRIMARY KEY (rule_id, stage));
CREATE TABLE dq_scorecard (stage TEXT, dimension TEXT, score REAL, PRIMARY KEY (stage, dimension));
"""

con = sqlite3.connect(DB_PATH)
con.executescript(DDL)
dm = dim_machine_all.drop(columns=["product_group", "model_desc"] + SPEC_COLS).copy()
spec_long = (dim_machine_all[["machine_key"] + SPEC_COLS]
             .melt(id_vars="machine_key", var_name="attribute", value_name="value"))
spec_long = spec_long[known(spec_long["value"])]
spec_long["attribute"] = spec_long["attribute"].str.lower()
for c in ("first_sale_date", "last_sale_date"):
    dm[c] = dm[c].dt.strftime("%Y-%m-%d")
fact = pd.DataFrame({
    "sales_id": fact_all["SalesID"], "machine_key": fact_all["machine_key"],
    "model_key_recorded": fact_all["model_key"].astype(int), "source_model_id": fact_all["ModelID"],
    "source_machine_id": fact_all["MachineID"], "sale_date": fact_all["saledate"].dt.strftime("%Y-%m-%d"),
    "sale_price": fact_all.get("SalePrice"), "hours_meter": fact_all["MachineHoursCurrentMeter"],
    "hours_raw": fact_all["hours_raw"], "hours_flag": fact_all["hours_flag"],
    "year_made_raw": fact_all["year_made_raw"], "year_flag": fact_all["year_flag"],
    "usage_band": fact_all["UsageBand"], "state": fact_all["state"], "auctioneer_id": fact_all["auctioneerID"],
    "datasource": fact_all["datasource"], "source_batch": fact_all["source_batch"], "er_rule": fact_all["er_rule"]})
tables = {
    "dim_model": dim_model_all, "xref_model": xref_model_all[["source_model_id", "model_key", "source_desc", "match_key", "n_sales"]],
    "dim_machine": dm, "machine_attribute": spec_long, "xref_machine": xref_machine_all,
    "fact_sale": fact,
    "dq_rule_result": pd.concat([rules_before, rules_after]).rename(columns={
        "dimensi": "dimension", "aturan": "description", "tahap": "stage", "diperiksa": "checked", "gagal": "failed"}),
    "dq_scorecard": score[["sebelum", "sesudah"]].reset_index(names="dimension")
                    .melt(id_vars="dimension", var_name="stage", value_name="score")[["stage", "dimension", "score"]],
}
for name, df_ in tables.items():
    df_.to_sql(name, con, if_exists="append", index=False, chunksize=50_000)
con.commit()
con.execute("VACUUM")
print(f"Database: {DB_PATH} ({DB_PATH.stat().st_size / 1e6:.0f} MB)")
tabs = [r[0] for r in con.execute("SELECT name FROM sqlite_master WHERE type='table' ORDER BY name")]
pd.DataFrame({"tabel": tabs,
              "kolom": [len(con.execute(f"PRAGMA table_info({t})").fetchall()) for t in tabs],
              "baris": [con.execute(f"SELECT COUNT(*) FROM {t}").fetchone()[0] for t in tabs]})

In [ ]:
checks = {
    "Pelanggaran foreign key": "SELECT COUNT(*) FROM pragma_foreign_key_check",
    "Model dengan > 1 deskripsi": "SELECT COUNT(*) FROM (SELECT model_key FROM dim_model GROUP BY model_key HAVING COUNT(DISTINCT model_desc) > 1)",
    "Golden record mesin tanpa model": "SELECT COUNT(*) FROM dim_machine WHERE model_key IS NULL",
    "Transaksi tanpa golden record": "SELECT COUNT(*) FROM fact_sale f LEFT JOIN dim_machine m USING(machine_key) WHERE m.machine_key IS NULL",
    "MachineID sumber tanpa crosswalk": "SELECT COUNT(DISTINCT source_machine_id) FROM fact_sale WHERE source_machine_id NOT IN (SELECT source_machine_id FROM xref_machine)",
    "Hour meter turun pada data bersih": """
        SELECT COUNT(*) FROM (SELECT hours_meter, LAG(hours_meter) OVER (PARTITION BY machine_key ORDER BY sale_date, sales_id) AS prev
        FROM fact_sale WHERE hours_meter IS NOT NULL) WHERE hours_meter < prev""",
}
display(pd.DataFrame([(k, con.execute(q).fetchone()[0]) for k, q in checks.items()], columns=["Cek integritas (SQL)", "Jumlah pelanggaran"]))
print("Contoh query bisnis — 5 model dengan transaksi terbanyak & umur rata-rata saat dijual:")
pd.read_sql("""
    SELECT m.model_desc, m.product_group_desc, COUNT(*) AS transaksi, ROUND(AVG(f.sale_price)) AS harga_rata2,
           ROUND(AVG(CAST(substr(f.sale_date, 1, 4) AS INT) - d.year_made), 1) AS umur_rata2_tahun
    FROM fact_sale f JOIN dim_machine d USING(machine_key) JOIN dim_model m ON m.model_key = d.model_key
    WHERE f.sale_price IS NOT NULL AND d.year_made IS NOT NULL
    GROUP BY m.model_key ORDER BY transaksi DESC LIMIT 5""", con)

In [ ]:
cmp.to_csv(CONFIG["OUT_DIR"] / "dq_rule_results.csv", index=False)
score.to_csv(CONFIG["OUT_DIR"] / "dq_scorecard.csv")
by_src.T.to_csv(CONFIG["OUT_DIR"] / "dq_by_datasource.csv")
con.close()
print("Ekspor CSV untuk dashboard Power BI (tahap berikutnya):", sorted(p.name for p in CONFIG["OUT_DIR"].glob("*.csv")))

## 4. Modeling — apakah master data yang bersih menghasilkan valuasi yang lebih akurat?
### 4.1 Desain eksperimen
* **Target:** `log(SalePrice)`; metrik **RMSLE** (metrik resmi kompetisi Blue Book) — error relatif harga.
* **Split berbasis waktu:** latih = penjualan sebelum 2011, uji = seluruh penjualan 2011 (±35 ribu transaksi).
* **Tanpa kebocoran:** master model & golden record untuk eksperimen ini dibangun **hanya dari data latih**
  (*as-of* 31 Des 2010), lalu dipakai untuk memperkaya transaksi uji — seperti sistem master data di dunia nyata.
* **Dua keluarga model:**
  * **Ridge (linear)** — mewakili model valuasi yang dapat dijelaskan (mis. tabel depresiasi/harga wajar).
  * **HistGradientBoosting** — model *black-box* yang kuat.
* **Dua versi data dengan transformasi yang SAMA** (log jam, tahun, tanggal jual, one-hot/ordinal), hanya kualitas datanya yang berbeda:
  * **Mentah** — kolom apa adanya (`YearMade = 1000`, jam = 0, `ModelID` sumber, teks tidak seragam).
  * **Master** — hasil standardisasi + master model kanonik + pengayaan dari golden record.
* Fitur `age = tahun jual − tahun buat` **sengaja tidak dipakai**: pada split waktu, fitur ini menggeser semua unit
  2011 "satu tahun lebih tua" sementara harga pasar 2011 naik, sehingga mengacaukan perbandingan (diuji saat eksplorasi).

In [ ]:
ML_TRAIN = train_raw["saledate"] < CONFIG["ML_CUTOFF"]
ML_TEST = ~ML_TRAIN
y_all = np.log1p(train_raw["SalePrice"].to_numpy())
y_tr, y_te = y_all[ML_TRAIN.to_numpy()], y_all[ML_TEST.to_numpy()]
print(f"Latih: {ML_TRAIN.sum():,} transaksi (≤ 2010) | Uji: {ML_TEST.sum():,} transaksi (2011)")

# ---- master AS-OF (hanya dari periode latih)
std_ml = std_train.copy()                                    # standardisasi bersifat per-baris (tanpa info masa depan)
asof = std_ml[ML_TRAIN.to_numpy()]
dim_model_asof, xref_asof = build_model_master(asof)
key_all = std_ml["desc_norm"] + "|" + std_ml["fiProductClassDesc"]
key_to_model = xref_asof.drop_duplicates("match_key").set_index("match_key")["model_key"]
std_ml["model_key"] = (std_ml["ModelID"].map(xref_asof.set_index("source_model_id")["model_key"])
                       .fillna(key_all.map(key_to_model)).fillna(std_ml["ModelID"]).astype(int))   # model baru -> ID sendiri
asof = std_ml[ML_TRAIN.to_numpy()]

# tahun dari golden record as-of: kunci entitas (MachineID + grup + base model) -> tahun valid paling sering
std_ml["k0"] = std_ml["MachineID"].astype(str) + "|" + std_ml["ProductGroup"] + "|" + std_ml["base_norm"].fillna("")
year_master = mode_by(asof.assign(k0=std_ml.loc[ML_TRAIN, "k0"]), "k0", "YearMade")["YearMade"]
model_year = asof.groupby("model_key")["YearMade"].median()
print(f"Master as-of: {len(dim_model_asof):,} model kanonik | {len(year_master):,} kunci mesin dengan tahun golden")


def features_raw(df):
    X = df.drop(columns=["SalesID", "SalePrice", "MachineID", "saledate", "source_batch"]).copy()
    X["sale_year"], X["sale_month"], X["sale_doy"] = df["saledate"].dt.year, df["saledate"].dt.month, df["saledate"].dt.dayofyear
    X["MachineHoursCurrentMeter"] = np.log1p(X["MachineHoursCurrentMeter"])
    return X


def features_master(d, steps=("validity", "standard", "model_master", "golden")):
    """Fitur dari data master. `steps` memungkinkan ablation kumulatif."""
    src = d if "standard" in steps else train_raw
    X = pd.DataFrame(index=d.index)
    X["sale_year"], X["sale_month"], X["sale_doy"] = d["saledate"].dt.year, d["saledate"].dt.month, d["saledate"].dt.dayofyear
    if "validity" in steps:
        X["YearMade"] = d["YearMade"]                                       # tahun tidak valid -> NaN
        X["hours"] = np.log1p(d["MachineHoursCurrentMeter"])                # jam 0 / mustahil -> NaN
    else:
        X["YearMade"] = train_raw["YearMade"]
        X["hours"] = np.log1p(train_raw["MachineHoursCurrentMeter"])
    if "golden" in steps:                                                   # tahun dari golden record mesin / median model
        from_master = X["YearMade"].isna() & d["k0"].map(year_master).notna()
        X["YearMade"] = X["YearMade"].fillna(d["k0"].map(year_master)).fillna(d["model_key"].map(model_year))
        X["year_source"] = np.select([d["YearMade"].notna(), from_master, X["YearMade"].notna()],
                                     ["sendiri", "golden mesin", "median model"], "tidak diketahui")
    cats = ["ProductGroup", "fiProductClassDesc", "fiBaseModel", "state", "datasource", "auctioneerID", "UsageBand",
            "ProductSize"] + SPEC_COLS
    for c in cats:
        X[c] = src[c].astype("string")
    if "standard" in steps:
        for c in ["tire_size_in", "pad_width_in", "stick_length_in", "blade_width_ft", "class_low", "class_high"]:
            X[c] = d[c]
        X["fiBaseModel"] = d["base_norm"]
    X["model"] = (d["model_key"] if "model_master" in steps else train_raw["ModelID"]).astype("string")
    if "golden" in steps:                                                   # spesifikasi tak diketahui -> konfigurasi umum model
        for c in SPEC_COLS + ["ProductSize"]:
            mm = mode_by(asof, "model_key", c)[c]
            unk = ~known(X[c])
            X[c] = X[c].where(~unk, d["model_key"].map(mm).astype("string")).fillna(X[c])
    return X


def to_hgb_matrix(X):
    X = X.copy()
    for c in X.columns:
        if X[c].dtype == object or str(X[c].dtype) in ("str", "string"):
            X[c] = X[c].astype("category").cat.codes.replace(-1, np.nan)
    return X.astype(float)


def rmsle(y, p):
    return float(np.sqrt(np.mean((y - p) ** 2)))


X_raw = features_raw(train_raw)
X_master = features_master(std_ml)
print(f"Fitur mentah: {X_raw.shape[1]} kolom | fitur master: {X_master.shape[1]} kolom")
print("Sumber tahun pembuatan pada data uji (versi master):")
X_master.loc[ML_TEST, "year_source"].value_counts(normalize=True).rename("proporsi").to_frame()

### 4.2 Model valuasi linear (Ridge)

In [ ]:
RIDGE_CATS = ["ProductGroup", "fiProductClassDesc", "fiBaseModel", "model", "Enclosure", "Hydraulics", "ProductSize",
              "Drive_System", "Tire_Size", "Track_Type", "Stick_Length", "Undercarriage_Pad_Width", "Blade_Type",
              "Ride_Control", "Coupler", "state", "datasource", "UsageBand"]


def ridge_fit_predict(X, extra_num=(), alphas=(1, 10, 100)):
    """Ridge sparse (solver lsqr). Alpha dipilih per versi data dengan validasi waktu di DALAM data latih
    (latih < 2010, validasi = 2010), lalu model dilatih ulang pada seluruh data latih."""
    num = ["YearMade", "hours", "sale_year", "sale_month"] + list(extra_num)
    cats = [c for c in RIDGE_CATS + (["year_source"] if "year_source" in X else []) if c in X]
    Xc = X[num + cats].copy()
    for c in cats:
        Xc[c] = Xc[c].astype(str)

    def make(alpha):
        pre = ColumnTransformer([
            ("num", make_pipeline(SimpleImputer(strategy="median", add_indicator=True), StandardScaler()), num),
            ("cat", OneHotEncoder(handle_unknown="infrequent_if_exist", min_frequency=20), cats)])
        return make_pipeline(pre, Ridge(alpha=alpha, solver="lsqr", tol=1e-4))

    Xtr = Xc[ML_TRAIN]
    inner = (train_raw.loc[ML_TRAIN, "saledate"] < "2010-01-01").to_numpy()
    scores = {a: rmsle(y_tr[~inner], make(a).fit(Xtr[inner], y_tr[inner]).predict(Xtr[~inner])) for a in alphas}
    best = min(scores, key=scores.get)
    return make(best).fit(Xtr, y_tr).predict(Xc[ML_TEST]), best


def ridge_raw_view(X):
    """Versi mentah memakai nama kolom yang sama dengan versi master."""
    V = X.rename(columns={"MachineHoursCurrentMeter": "hours"}).copy()
    V["model"] = train_raw["ModelID"].astype(str)
    return V


EXTRA = ["class_low", "class_high", "tire_size_in", "pad_width_in", "stick_length_in", "blade_width_ft"]
pred = {}
t = time.time()
pred[("Ridge", "Mentah")], a_raw = ridge_fit_predict(ridge_raw_view(X_raw))
pred[("Ridge", "Master")], a_mst = ridge_fit_predict(X_master, EXTRA)
print(f"Ridge mentah RMSLE = {rmsle(y_te, pred[('Ridge', 'Mentah')]):.4f} (alpha {a_raw}) | "
      f"Ridge master RMSLE = {rmsle(y_te, pred[('Ridge', 'Master')]):.4f} (alpha {a_mst}) | {time.time() - t:.0f} detik")

#### Ablation kumulatif: langkah pembersihan mana yang paling berdampak?

In [ ]:
ablation_steps = [
    ("0. Data mentah", None),
    ("1. + aturan validitas (tahun & hour meter)", ("validity",)),
    ("2. + standardisasi (teks, Excel, Unknown/N-A, satuan, kelas)", ("validity", "standard")),
    ("3. + master model kanonik", ("validity", "standard", "model_master")),
    ("4. + golden record (tahun mesin, konfigurasi model)", ("validity", "standard", "model_master", "golden")),
]
abl = []
for name, steps in ablation_steps:
    if steps is None:
        p = pred[("Ridge", "Mentah")]
    elif steps == ablation_steps[-1][1]:
        p = pred[("Ridge", "Master")]
    else:
        Xs = features_master(std_ml, steps)
        p, _ = ridge_fit_predict(Xs, EXTRA if "standard" in steps else ())
    abl.append({"langkah": name, "RMSLE": rmsle(y_te, p)})
abl = pd.DataFrame(abl)
abl["perubahan vs sebelumnya"] = abl["RMSLE"].diff()
abl["perubahan kumulatif"] = abl["RMSLE"] / abl["RMSLE"].iloc[0] - 1
display(abl)

fig, ax = plt.subplots(figsize=(10, 4.6))
prev = abl["RMSLE"].iloc[0]
for i, r in abl.iterrows():
    if i == 0:
        ax.bar(i, r.RMSLE, color=C["muted"], width=0.6)
    else:
        lo, hi = sorted([prev, r.RMSLE])
        ax.bar(i, hi - lo, bottom=lo, color=C["green"] if r.RMSLE < prev else C["bad"], width=0.6)
        dlt = r.RMSLE - prev
        ax.text(i, hi + 0.003, "≈ 0" if abs(dlt) < 0.0005 else f"{dlt:+.3f}", ha="center", fontsize=9, color=C["ink"])
    prev = r.RMSLE
ax.bar(len(abl), abl["RMSLE"].iloc[-1], color=C["blue"], width=0.6)
ax.text(0, abl["RMSLE"].iloc[0] + 0.003, f"{abl['RMSLE'].iloc[0]:.3f}", ha="center", fontsize=9)
ax.text(len(abl), abl["RMSLE"].iloc[-1] + 0.003, f"{abl['RMSLE'].iloc[-1]:.3f}", ha="center", fontsize=9)
ax.set_xticks(range(len(abl) + 1), ["Mentah", "Validitas", "Standardisasi", "Master\nmodel", "Golden\nrecord", "Master\n(akhir)"])
ax.set_ylim(abl["RMSLE"].min() * 0.85, abl["RMSLE"].max() * 1.05)
ax.set_ylabel("RMSLE uji (lebih kecil lebih baik)")
ax.set_title("Kontribusi tiap langkah master data pada error model valuasi (Ridge)")
subtitle(ax, f"Total perbaikan {-abl['perubahan kumulatif'].iloc[-1]:.0%}; hijau = error turun")
ax.grid(axis="x", visible=False)
plt.tight_layout()
plt.show()

### 4.3 Model *gradient boosting* (3 seed untuk mengukur noise)

In [ ]:
H_raw, H_master = to_hgb_matrix(X_raw), to_hgb_matrix(X_master)
hgb_rows = []
for version, H in (("Mentah", H_raw), ("Master", H_master)):
    preds = []
    for seed in (0, 1, 2):
        m = HistGradientBoostingRegressor(max_iter=400, learning_rate=0.1, max_leaf_nodes=63, min_samples_leaf=20,
                                          early_stopping=False, random_state=seed)
        m.fit(H[ML_TRAIN], y_tr)
        p = m.predict(H[ML_TEST])
        preds.append(p)
        hgb_rows.append({"versi": version, "seed": seed, "RMSLE": rmsle(y_te, p)})
    pred[("HistGradientBoosting", version)] = np.mean(preds, axis=0)
hgb = pd.DataFrame(hgb_rows)
display(hgb.pivot(index="seed", columns="versi", values="RMSLE"))
noise = hgb.groupby("versi")["RMSLE"].std().max()
print(f"Noise antar-seed (std): {noise:.4f}")

## 5. Evaluation
### 5.1 Akurasi pada data uji 2011 + interval kepercayaan (paired bootstrap)

In [ ]:
rng = np.random.default_rng(CONFIG["SEED"])
B = 1000
idx = rng.integers(0, len(y_te), size=(B, len(y_te)))
res_rows = []
for fam in ("Ridge", "HistGradientBoosting"):
    e_raw = (y_te - pred[(fam, "Mentah")]) ** 2
    e_mst = (y_te - pred[(fam, "Master")]) ** 2
    diff = np.sqrt(e_raw[idx].mean(axis=1)) - np.sqrt(e_mst[idx].mean(axis=1))
    res_rows.append({"model": fam, "RMSLE mentah": np.sqrt(e_raw.mean()), "RMSLE master": np.sqrt(e_mst.mean()),
                     "perubahan error": np.sqrt(e_mst.mean()) / np.sqrt(e_raw.mean()) - 1,
                     "CI95 penurunan (bawah)": np.percentile(diff, 2.5), "CI95 penurunan (atas)": np.percentile(diff, 97.5)})
results = pd.DataFrame(res_rows)
results["signifikan?"] = np.where(results["CI95 penurunan (bawah)"] > 0, "ya, lebih baik",
                                  np.where(results["CI95 penurunan (atas)"] < 0, "lebih buruk", "tidak berbeda"))
results

### 5.2 Error per segmen data bermasalah
Di mana master data paling berdampak? Segmen didefinisikan dari **data mentah** transaksi uji.

In [ ]:
te_raw = train_raw[ML_TEST]
train_model_counts = train_raw.loc[ML_TRAIN, "ModelID"].value_counts()
collided = set(split_ids[split_ids > 1].index)
segments = {
    "Semua transaksi": np.ones(len(te_raw), bool),
    "YearMade placeholder/invalid": ~te_raw["YearMade"].between(CONFIG["YEAR_MIN"], te_raw["saledate"].dt.year).to_numpy(),
    "Hour meter 0/mustahil": (te_raw["MachineHoursCurrentMeter"].eq(0) |
                              te_raw["MachineHoursCurrentMeter"].gt(CONFIG["HOURS_MAX"])).to_numpy(),
    "Model langka/baru (≤ 10 transaksi latih)": te_raw["ModelID"].map(train_model_counts).fillna(0).le(10).to_numpy(),
    "MachineID bertabrakan (ID dipakai ulang)": te_raw["MachineID"].isin(collided).to_numpy(),
}
seg_rows = []
for seg, mask in segments.items():
    for fam in ("Ridge", "HistGradientBoosting"):
        r = {"segmen": seg, "n": int(mask.sum()), "model": fam}
        for v in ("Mentah", "Master"):
            r[v] = rmsle(y_te[mask], pred[(fam, v)][mask])
        r["perubahan"] = r["Master"] / r["Mentah"] - 1
        seg_rows.append(r)
seg = pd.DataFrame(seg_rows)
display(seg)

fig, axes = plt.subplots(1, 2, figsize=(15, 4.8), sharey=True)
for ax, fam in zip(axes, ("Ridge", "HistGradientBoosting")):
    d = seg[seg["model"] == fam].reset_index(drop=True)
    yy = np.arange(len(d))[::-1]
    ax.barh(yy + 0.2, d["Mentah"], height=0.4, color=C["muted"], label="Data mentah")
    ax.barh(yy - 0.2, d["Master"], height=0.4, color=C["blue"], label="Data master")
    for y_, r in zip(yy, d.itertuples()):
        ax.text(max(r.Mentah, r.Master) + 0.005, y_, f"{r.perubahan:+.0%}", va="center", fontsize=9,
                color=C["good"] if r.perubahan < -0.01 else (C["bad"] if r.perubahan > 0.01 else C["ink2"]))
    ax.set_yticks(yy, [f"{s}\n(n={n:,})" for s, n in zip(d["segmen"], d["n"])], fontsize=8.5)
    ax.set_xlabel("RMSLE uji 2011")
    ax.set_title(fam, fontsize=11)
    ax.grid(axis="y", visible=False)
axes[0].legend(loc="lower right", fontsize=8.5)
sr = seg.set_index(["model", "segmen"])["perubahan"]
subtitle(axes[0], f"Perbaikan terbesar pada model langka/baru ({sr['Ridge', 'Model langka/baru (≤ 10 transaksi latih)']:+.0%}); "
                  f"segmen tanpa tahun valid memburuk ({sr['Ridge', 'YearMade placeholder/invalid']:+.0%})")
subtitle(axes[1], f"Juga membaik, terbesar pada transaksi tanpa tahun valid "
                  f"({sr['HistGradientBoosting', 'YearMade placeholder/invalid']:+.0%})")
plt.tight_layout()
plt.show()

### 5.3 Dampak pada KPI bisnis armada (tanpa ML)
Bahkan tanpa model, master data yang kotor langsung mendistorsi angka yang dilaporkan ke manajemen.

In [ ]:
gm_all = dim_machine
last_sale_year = gm_all["last_sale_date"].dt.year
raw_last = train_raw.sort_values("saledate").groupby("MachineID").last()
kpi = pd.DataFrame([
    ("Jumlah unit (populasi armada)", f"{train_raw['MachineID'].nunique():,}", f"{len(gm_all):,}",
     "ID dipakai ulang → unit tercampur dalam satu riwayat"),
    ("Jumlah model", f"{train_raw['ModelID'].nunique():,}", f"{len(dim_model):,}", "duplikat ModelID digabung"),
    ("Rata-rata tahun pembuatan unit", f"{raw_last['YearMade'].mean():,.0f}", f"{gm_all['year_made'].mean():,.0f}",
     "placeholder 1000 menarik rata-rata ke abad ke-19"),
    ("Rata-rata umur unit saat transaksi terakhir (tahun)",
     f"{(raw_last['saledate'].dt.year - raw_last['YearMade']).mean():,.1f}",
     f"{(last_sale_year - gm_all['year_made']).mean():,.1f}", ""),
    ("Median hour meter (transaksi terisi)", f"{train_raw['MachineHoursCurrentMeter'].median():,.0f}",
     f"{fact_train['MachineHoursCurrentMeter'].median():,.0f}", "nilai 0 diperlakukan sebagai 'tidak diketahui'"),
    ("Rata-rata hour meter", f"{train_raw['MachineHoursCurrentMeter'].mean():,.0f}",
     f"{fact_train['MachineHoursCurrentMeter'].mean():,.0f}", "outlier jutaan jam dikarantina"),
], columns=["KPI", "Data mentah", "Golden record", "Penyebab perbedaan"])
kpi

### 5.4 Evaluasi terhadap kriteria sukses

In [ ]:
rb, ra = rules_before.set_index("rule_id"), rules_after.set_index("rule_id")
ridge_r = results.set_index("model").loc["Ridge"]
hgb_r = results.set_index("model").loc["HistGradientBoosting"]
con = sqlite3.connect(DB_PATH)
fk_ok = con.execute("SELECT COUNT(*) FROM pragma_foreign_key_check").fetchone()[0] == 0
orphans = con.execute("SELECT COUNT(*) FROM fact_sale f LEFT JOIN dim_machine m USING(machine_key) "
                      "WHERE m.machine_key IS NULL").fetchone()[0]
con.close()
criteria = pd.DataFrame([
    ("S1", "Validitas, konsistensi, keunikan ≥ 99%",
     ", ".join(f"{d} {score.loc[d, 'sesudah']:.1%}" for d in ["Validitas", "Konsistensi", "Keunikan"]),
     all(score.loc[d, "sesudah"] >= 0.99 for d in ["Validitas", "Konsistensi", "Keunikan"])),
    ("S2", "Kelengkapan naik tanpa imputasi palsu",
     f"{score.loc['Kelengkapan', 'sebelum']:.1%} → {score.loc['Kelengkapan', 'sesudah']:.1%}",
     score.loc["Kelengkapan", "sesudah"] > score.loc["Kelengkapan", "sebelum"]),
    ("S3", "Satu nilai per atribut per mesin & model (R04–R07)",
     f"pass rate {ra.loc[['R04', 'R05', 'R06', 'R07'], 'pass_rate'].min():.1%}",
     ra.loc[["R04", "R05", "R06", "R07"], "pass_rate"].min() == 1),
    ("S4", "Ketertelusuran penuh (FK & crosswalk)", f"FK valid: {fk_ok}, transaksi yatim: {orphans}",
     fk_ok and orphans == 0),
    ("S5", "Model valuasi linear lebih akurat & signifikan",
     f"RMSLE {ridge_r['RMSLE mentah']:.3f} → {ridge_r['RMSLE master']:.3f} ({ridge_r['perubahan error']:+.1%}), "
     f"CI95 [{ridge_r['CI95 penurunan (bawah)']:.3f}, {ridge_r['CI95 penurunan (atas)']:.3f}]",
     ridge_r["CI95 penurunan (bawah)"] > 0),
    ("S6", "Gradient boosting tidak memburuk (≤ noise antar-seed)",
     f"RMSLE {hgb_r['RMSLE mentah']:.4f} → {hgb_r['RMSLE master']:.4f}, noise {noise:.4f}",
     hgb_r["RMSLE master"] - hgb_r["RMSLE mentah"] <= max(noise, 0.002)),
], columns=["ID", "Kriteria", "Hasil", "Lolos"])
criteria["Status"] = np.where(criteria["Lolos"], "LOLOS", "TIDAK LOLOS")
print(f"Total waktu notebook: {(time.time() - T0) / 60:.1f} menit")
criteria.drop(columns="Lolos")

### 5.5 Kesimpulan

**Master data**
1. **Identitas unit tidak bisa dipercaya begitu saja.** 5.214 `MachineID` ternyata mewakili lebih dari satu alat fisik
   (contoh ekstrem: satu ID = 13 unit loader & excavator). *Entity resolution* menghasilkan 347.399 golden record
   dari 341.027 ID sumber, masing-masing dengan skor *confidence* dan *crosswalk* ke ID asal.
2. **Validitas & konsistensi dapat dipulihkan** (84% → 100% dan 94% → 99,9%): placeholder tahun, hour meter mustahil
   dan yang turun dikarantina dengan *flag* (nilai asli tetap disimpan), 78 ModelID duplikat digabung, korupsi Excel
   (`#NAME?`, `214E4` → `2.14E+06`) direkonstruksi.
3. **Kelengkapan tidak bisa "dibersihkan".** Spesifikasi yang tidak diketahui tidak pernah diketahui di transaksi lain
   mesin yang sama; cakupan 56,7% hanya bisa naik lewat perbaikan proses input (kriteria S2 tidak lolos — dilaporkan
   apa adanya, bukan ditutupi dengan imputasi).
4. **Masalah berakar pada sumber tertentu**: hour meter datasource 136 hanya 20% valid; datasource 172 paling sering
   memakai ulang ID mesin dan mencatat hour meter yang turun.

**Dampak bisnis**
5. **KPI armada terdistorsi parah tanpa master data**: umur rata-rata unit 106,7 tahun (mentah) vs 10,3 tahun (golden);
   median hour meter 0 vs 3.092 jam.
6. **Model valuasi lebih akurat**: Ridge −9,5% RMSLE (CI 95% signifikan), terbesar pada model langka/baru (−29%) berkat
   master model kanonik; gradient boosting pun membaik −1,9% (9× lebih besar dari noise antar-seed).
7. **Langkah paling berdampak** adalah aturan validitas (tahun & hour meter), disusul golden record.
8. **Temuan jujur**: pada transaksi tanpa tahun valid, Ridge sedikit memburuk (+4%) karena tahun diisi median model —
   unit tanpa tahun cenderung lebih tua (data hilang tidak acak). Ini memperkuat rekomendasi: tahun pembuatan harus
   menjadi *field* wajib yang diverifikasi dari plat nomor seri.

### 5.6 Rekomendasi tata kelola data (*data governance*)
| Prioritas | Rekomendasi | Aturan terkait |
|---|---|---|
| 1 | Nomor unit unik per alat fisik; larang pemakaian ulang ID (validasi grup produk & model saat input) | R06–R08 |
| 2 | Tahun pembuatan & hour meter wajib, dengan validasi rentang dan cek "tidak turun" di sistem entri | R01–R03, R14 |
| 3 | Ekspor/impor data tanpa Excel (atau kolom teks dipaksa *text format*) untuk mencegah `#NAME?` & notasi ilmiah | R09 |
| 4 | Daftar nilai baku (*reference data*) untuk atribut spesifikasi beserta satuannya | R10, R13 |
| 5 | Jalankan katalog aturan ini terjadwal pada setiap batch baru; pantau scorecard per sumber data | semua |

**Keterbatasan**: dataset publik lelang AS (proksi data armada), tidak ada *ground truth* identitas mesin sehingga
akurasi *entity resolution* tidak dapat diukur langsung (diukur lewat konsistensi & dampak hilir); ambang aturan
(mis. 100.000 jam, toleransi tahun ±1) adalah asumsi yang dapat dikonfigurasi di `CONFIG`.

**Tahap berikutnya**: dashboard Power BI *Data Quality Scorecard* dari tabel `dq_rule_result`, `dq_scorecard`, dan
`outputs/*.csv`.